# LangChain Agent Builder - 使用框架构建Agent

## 学习目标
- 使用 `create_react_agent` 快速构建 ReAct Agent
- 配置 `AgentExecutor` 实现执行控制
- 使用 `@tool` 装饰器创建 LangChain 风格工具
- 对比从零实现 vs 使用框架
- 决策树：何时使用框架 vs 自定义
- 高级：自定义中间件（监控、重试）和配置

## 1. 环境准备

需要安装的依赖：
```bash
pip install langchain langchain-openai langgraph
```

> 本 Notebook 在没有安装 LangChain 的环境中也能运行——代码内置了模拟模式（HAS_LANGCHAIN=False）。

In [ ]:
# 基础导入
import os, sys, time, json, math, re
from typing import Any, Optional, Callable
from pprint import pprint

print("基础库导入成功")

In [ ]:
# LangChain 导入
try:
    from langchain_core.tools import tool
    from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
    from langchain_openai import ChatOpenAI
    from langgraph.prebuilt import create_react_agent
    print("LangChain 库导入成功")
    HAS_LANGCHAIN = True
except ImportError as e:
    print(f"[模拟模式] LangChain 导入失败 ({e})，将使用模拟实现进行演示")
    print("如需实际运行: pip install langchain langchain-openai langgraph")
    HAS_LANGCHAIN = False

## 2. 使用 @tool 装饰器创建工具

LangChain 提供了 `@tool` 装饰器，可以快速将 Python 函数转换为 Agent 可用的工具。
装饰器会自动从函数的 docstring 和类型注解中提取工具描述和参数 schema。

以下创建 **4 个工具**：天气查询、计算器、知识搜索、文件读取。

In [ ]:
# ============================================================
# 定义 4 个工具（使用 @tool 装饰器 - LangChain 风格）
# ============================================================

from langchain_core.tools import tool


@tool
def weather_tool(city: str, units: str = "celsius") -> str:
    """查询指定城市的天气信息。

    Args:
        city: 城市名称，如 '北京'、'上海'、'广州'、'深圳'
        units: 温度单位，'celsius' 或 'fahrenheit'

    Returns:
        该城市的天气描述字符串
    """
    weather_db = {
        "北京": {"condition": "晴", "temp": 22, "humidity": 45},
        "上海": {"condition": "多云", "temp": 28, "humidity": 70},
        "广州": {"condition": "雷阵雨", "temp": 32, "humidity": 85},
        "深圳": {"condition": "多云转晴", "temp": 30, "humidity": 75},
        "成都": {"condition": "阴", "temp": 25, "humidity": 60},
    }
    city_data = weather_db.get(city, {"condition": "未知", "temp": 20, "humidity": 50})
    temp = city_data["temp"]
    if units == "fahrenheit":
        temp = round(temp * 9 / 5 + 32, 1)
        unit_str = "°F"
    else:
        unit_str = "°C"
    return f"{city}天气: {city_data['condition']}，温度 {temp}{unit_str}，湿度 {city_data['humidity']}%，查询时间 {time.strftime('%Y-%m-%d %H:%M:%S')}"


@tool
def calculator_tool(formula: str) -> str:
    """安全地计算数学表达式的结果。

    Args:
        formula: 数学表达式字符串，支持 +, -, *, /, **, sqrt, sin, cos, log, pi, e

    Returns:
        计算结果描述字符串
    """
    safe_names = {
        'sqrt': math.sqrt, 'sin': math.sin, 'cos': math.cos,
        'tan': math.tan, 'log': math.log, 'log10': math.log10,
        'exp': math.exp, 'pi': math.pi, 'e': math.e,
        'abs': abs, 'pow': pow, 'round': round,
        'ceil': math.ceil, 'floor': math.floor,
    }
    try:
        result = eval(formula, {"__builtins__": {}}, safe_names)
        return f"计算结果: {formula} = {result}"
    except SyntaxError as e:
        return f"表达式语法错误: {e}"
    except Exception as e:
        return f"计算错误: {e}"


@tool
def search_tool(query: str, max_results: int = 5) -> str:
    """搜索知识库获取信息。

    Args:
        query: 搜索查询关键词
        max_results: 最大返回结果数，默认5

    Returns:
        搜索结果描述字符串
    """
    knowledge_base = {
        "人工智能": [
            "AI是计算机科学的重要分支，研究如何创建能够模拟人类智能的系统",
            "AI包含多个子领域：机器学习、深度学习、NLP、计算机视觉",
        ],
        "机器学习": [
            "机器学习是AI的核心方法，让计算机从数据中自动学习模式和规律",
            "三大学习范式：监督学习、无监督学习、强化学习",
        ],
        "深度学习": [
            "深度学习使用多层神经网络处理复杂的非线性问题",
            "CNN用于图像，RNN/LSTM用于序列，Transformer用于NLP",
        ],
        "Agent": [
            "AI Agent是能自主感知环境、做出决策并执行行动的智能实体",
            "ReAct模式结合了推理(Reasoning)和行动(Acting)，是主流架构",
        ],
    }
    results = []
    ql = query.lower()
    for topic, entries in knowledge_base.items():
        if ql in topic.lower() or any(ql in e.lower() for e in entries):
            for entry in entries:
                results.append(f"[{topic}] {entry}")
    if not results:
        for topic, entries in knowledge_base.items():
            words = set(ql.split())
            if words & set(topic.lower().split()):
                for entry in entries[:max_results]:
                    results.append(f"[{topic}] {entry}")
    results = results[:max_results]
    if results:
        return f"搜索 '{query}' 找到 {len(results)} 条结果:\n" + "\n".join(results)
    else:
        return f"未找到关于 '{query}' 的信息"


@tool
def file_reader_tool(path: str) -> str:
    """读取文件内容（最大1MB）。

    Args:
        path: 文件的完整路径

    Returns:
        文件内容或错误信息
    """
    if not os.path.exists(path):
        return f"错误: 文件不存在 - {path}"
    file_size = os.path.getsize(path)
    if file_size > 1024 * 1024:
        return f"错误: 文件过大({file_size} bytes)，超过1MB限制"
    try:
        with open(path, 'r', encoding='utf-8') as f:
            content = f.read()
        lines = len(content.split('\n'))
        words = len(content.split())
        return f"文件 '{path}': {lines}行, {words}词, {len(content)}字符. 内容:\n{content[:2000]}"
    except UnicodeDecodeError:
        return f"错误: 无法以UTF-8解码文件 '{path}'"
    except Exception as e:
        return f"错误: {e}"


# 将工具放入列表
tools = [weather_tool, calculator_tool, search_tool, file_reader_tool]

print(f"已创建 {len(tools)} 个工具:")
for t in tools:
    print(f"  - {t.name}: {t.description[:60]}...")

## 3. 使用 create_react_agent 构建 Agent

`create_react_agent` 是 LangGraph 提供的便捷函数，它能：
- 自动构建 ReAct 循环（Thought -> Action -> Observation）
- 处理工具调用的解析和执行
- 管理对话状态和消息历史

只需传入模型（ChatOpenAI）、工具列表、可选的系统提示词即可。

In [ ]:
# 构建 ReAct Agent

if HAS_LANGCHAIN:
    # 初始化模型
    model = ChatOpenAI(
        model="gpt-4",
        temperature=0,
    )

    # 系统提示词
    system_prompt = "你是一个有用的AI助手，可以使用工具来完成任务。"

    # 创建 ReAct Agent
    agent = create_react_agent(
        model=model,
        tools=tools,
    )

    print("ReAct Agent 创建成功!")
    print(f"模型: gpt-4")
    print(f"工具数: {len(tools)}")
else:
    print("[模拟] ReAct Agent 创建成功!")
    print("代码结构:")
    print("  agent = create_react_agent(model=ChatOpenAI(...), tools=[...])")
    print("注意: 需要安装 langchain 并设置 API 密钥才能实际运行")

## 4. AgentExecutor 执行控制

`AgentExecutor` 提供对 Agent 执行过程的精细控制：

| 参数 | 类型 | 说明 |
|------|------|------|
| `max_iterations` | int | 最大迭代步数，防止无限循环 |
| `max_execution_time` | float | 最大执行时间（秒） |
| `return_intermediate_steps` | bool | 是否返回中间推理步骤 |
| `handle_parsing_errors` | bool/str/Callable | 解析错误处理策略 |

### handle_parsing_errors 的三种模式

1. **`handle_parsing_errors=True`**: 当模型输出无法解析时，返回原始输出作为降级方案（fallback），Agent 继续运行
2. **`handle_parsing_errors="raise"`**: 当解析失败时立即抛出异常，停止 Agent 执行——适合开发调试
3. **`handle_parsing_errors=custom_function`**: 传入自定义函数处理错误，例如尝试修复格式并重试

In [ ]:
# ============================================================
# handle_parsing_errors 三种模式演示
# ============================================================

def custom_parse_error_handler(error: Exception) -> str:
    """
    自定义解析错误处理函数 —— 模式 3。
    当 Agent 输出的工具调用格式无法解析时，
    返回一条帮助信息让 Agent 纠正输出。
    """
    return (
        f"解析工具调用时出错: {str(error)[:200]}\n"
        f"请确保使用正确的工具调用格式。\n"
        f"可用工具: weather_tool(city, units), calculator_tool(formula), "
        f"search_tool(query, max_results), file_reader_tool(path)"
    )


def demonstrate_parsing_error_modes():
    """
    演示三种 handle_parsing_errors 模式的行为差异。
    模拟模型输出了一个格式错误的工具调用。
    """
    malformed_output = "weather(city=北京 units=celsius)"  # 故意格式错误

    print("=" * 60)
    print("handle_parsing_errors 三种模式对比")
    print("=" * 60)
    print(f"\n模拟错误输出: {malformed_output}")

    # 模式 1: True — 返回原始输出作为 fallback
    print("\n[模式 1] handle_parsing_errors=True:")
    print("  行为: 解析失败时返回原始文本，Agent 继续运行")
    print(f"  降级输出: {malformed_output}")
    print("  适用: 生产环境，容错优先")

    # 模式 2: "raise" — 抛出异常
    print("\n[模式 2] handle_parsing_errors='raise':")
    print("  行为: 解析失败时立即抛出 OutputParserException")
    print("  异常: OutputParserException: Could not parse LLM output")
    print("  适用: 开发/调试，快速发现问题")

    # 模式 3: custom_function — 自定义修复
    print("\n[模式 3] handle_parsing_errors=custom_function:")
    error = Exception("Could not parse: invalid argument separator")
    suggestion = custom_parse_error_handler(error)
    print(f"  行为: 调用自定义函数处理错误")
    print(f"  返回给模型: {suggestion[:100]}...")
    print("  适用: 希望自动修复并重试的场景")

    print("\n总结:")
    print("  True   = 容错模式（生产推荐）")
    print("  raise  = 严格模式（开发调试）")
    print("  custom = 智能修复（高级场景）")


demonstrate_parsing_error_modes()

In [ ]:
# ============================================================
# AgentExecutor —— 完整的运行函数
# ============================================================

def run_agent_with_executor(
    task: str,
    max_iterations: int = 10,
    max_time: float = 60.0,
    debug: bool = False,
) -> dict:
    """
    使用执行控制参数运行 Agent。

    Args:
        task: 用户任务描述
        max_iterations: 最大迭代步数，防止无限循环
        max_time: 最大执行时间(秒)
        debug: 是否返回中间推理步骤

    Returns:
        包含执行结果和元信息的字典
    """
    start_time = time.time()
    intermediate_steps = []

    if not HAS_LANGCHAIN:
        # 模拟执行
        elapsed = time.time() - start_time
        return {
            "task": task,
            "result": f"[模拟结果] 关于'{task}'的分析完成。共使用{len(tools)}个工具。",
            "iterations": 3,
            "execution_time": round(elapsed, 2),
            "success": True,
            "intermediate_steps": [],
        }

    try:
        inputs = {"messages": [HumanMessage(content=task)]}

        configured_agent = agent.with_config({
            "recursion_limit": max_iterations,
        })

        if debug:
            steps = []
            for step_output in configured_agent.stream(inputs):
                steps.append(step_output)
                intermediate_steps.append(step_output)
                if time.time() - start_time > max_time:
                    print(f"[超时] 执行超过 {max_time} 秒，强制停止")
                    break
            if steps:
                last_step = steps[-1]
                agent_messages = last_step.get("agent", {}).get("messages", [])
                if agent_messages:
                    final_message = agent_messages[-1]
                    result = final_message.content if hasattr(final_message, 'content') else str(final_message)
                else:
                    result = "Agent 未返回最终消息"
            else:
                result = "Agent 执行未产生输出"
        else:
            result_state = configured_agent.invoke(inputs)
            messages = result_state.get("messages", [])
            if messages:
                result = messages[-1].content
            else:
                result = "无响应"

        elapsed = time.time() - start_time
        return {
            "task": task,
            "result": result,
            "iterations": len(intermediate_steps),
            "execution_time": round(elapsed, 2),
            "success": True,
            "intermediate_steps": intermediate_steps if debug else [],
        }

    except Exception as e:
        elapsed = time.time() - start_time
        return {
            "task": task,
            "result": f"执行失败: {e}",
            "iterations": len(intermediate_steps),
            "execution_time": round(elapsed, 2),
            "success": False,
            "error": str(e),
        }


print("AgentExecutor 运行函数定义完成")

In [ ]:
# 运行演示任务

test_tasks = [
    "查询北京的天气",
    "计算 sqrt(256) + log(100) 的结果",
    "搜索关于人工智能的信息",
    "查询上海天气，并计算当地华氏温度",
]

results = []
for task in test_tasks:
    print(f"\n{'='*60}")
    print(f"任务: {task}")
    print(f"{'='*60}")
    result = run_agent_with_executor(
        task=task, max_iterations=10, max_time=30.0, debug=False,
    )
    results.append(result)
    print(f"结果: {result['result'][:200]}...")
    print(f"耗时: {result['execution_time']}s, 成功: {result['success']}")

print(f"\n{'='*60}")
print(f"共执行 {len(results)} 个任务")
print(f"成功: {sum(1 for r in results if r['success'])}/{len(results)}")
print(f"平均耗时: {sum(r['execution_time'] for r in results)/len(results):.2f}s")

## 5. 完整示例：研究型 Agent

构建一个研究型 Agent，综合使用搜索、计算、文件工具完成研究任务。

In [ ]:
# 创建测试文件供 file_reader_tool 使用
test_file_path = "research_notes.txt"
with open(test_file_path, 'w', encoding='utf-8') as f:
    f.write("研究笔记 - AI Agent 开发\n")
    f.write("=" * 40 + "\n")
    f.write("1. Agent架构选择: ReAct 模式适合大多数场景\n")
    f.write("2. 工具设计: 每个工具应该职责单一、描述清晰\n")
    f.write("3. 记忆管理: 分层记忆(短期+长期+情景)是生产级方案\n")
    f.write("4. 错误处理: 实现多层回退策略(Hard->Soft->Regex)\n")
    f.write("5. 安全: 工具执行需要沙箱隔离和超时控制\n")

print(f"测试文件已创建: {test_file_path}")
print(f"内容预览:")
with open(test_file_path, 'r', encoding='utf-8') as f:
    print(f.read())

In [ ]:
# 研究型Agent演示

research_task = """
请完成以下研究任务:
1. 搜索关于AI Agent的信息
2. 计算 2^10 + sqrt(1024) 的值
3. 读取 research_notes.txt 文件
4. 根据搜索结果和文件内容，总结Agent开发的关键要点
"""

print("执行研究型任务...")
print(f"任务: {research_task[:100]}...")

research_result = run_agent_with_executor(
    task=research_task,
    max_iterations=15,
    max_time=90.0,
    debug=False,
)

print(f"\n结果: {research_result['result'][:500]}...")
print(f"迭代次数: {research_result['iterations']}")
print(f"执行时间: {research_result['execution_time']}s")

In [ ]:
# 清理测试文件
if os.path.exists(test_file_path):
    os.remove(test_file_path)
    print(f"已清理: {test_file_path}")

## 6. 对比: 从零实现 vs LangChain 框架

### 代码行数对比表

In [ ]:
# ============================================================
# 代码行数对比表（可以运行的代码，非死数据）
# ============================================================

comparison_data = {
    "代码行数": {
        "从零实现 (Phase 06-02)": "~150-200 行",
        "LangChain create_react_agent": "~30-50 行",
        "代码节省": "~70%",
    },
    "核心功能对比": {
        "从零实现": "ReAct循环、工具执行、基本错误处理",
        "LangChain": "ReAct循环、工具执行、流式输出、中间件、自动追踪、多种Agent模式、状态管理、错误恢复",
    },
    "学习曲线": {
        "从零实现": "陡峭 — 需要理解所有底层细节",
        "LangChain": "平缓 — 专注业务逻辑，框架处理基础设施",
    },
    "灵活性": {
        "从零实现": "极高 — 完全控制每个细节",
        "LangChain": "中等 — 遵循框架约定，自定义需深入理解",
    },
    "错误处理": {
        "从零实现": "需自行实现重试、回退、超时",
        "LangChain": "内置 handle_parsing_errors、max_iterations、回调系统",
    },
    "流式输出": {
        "从零实现": "需手动实现 yield/生成器",
        "LangChain": "内置 .stream() 方法，开箱即用",
    },
    "追踪/监控": {
        "从零实现": "需自行对接日志系统",
        "LangChain": "LangSmith 自动追踪，回调系统可扩展",
    },
    "并行工具调用": {
        "从零实现": "需手动实现 asyncio 调度",
        "LangChain": "内置并行调度（依赖自动检测）",
    },
    "生产就绪度": {
        "从零实现": "低 — 需自己实现监控、日志、追踪",
        "LangChain": "高 — 内置追踪、流式、回调、错误恢复",
    },
    "适用场景": {
        "从零实现": "学习目的、高度定制、资源受限环境、无外部依赖要求",
        "LangChain": "快速原型、生产部署、需要生态集成、团队协作",
    },
}

print("=" * 60)
print("从零实现 vs LangChain 框架 — 全面特性对比")
print("=" * 60)

for category, items in comparison_data.items():
    print(f"\n  [{category}]")
    for key, value in items.items():
        print(f"    {key}: {value}")

print("\n" + "=" * 60)
print("核心建议: 学习时从零实现，生产时使用框架")
print("=" * 60)

### 决策树：何时用框架 vs 自定义

面对一个新项目时，按以下决策树选择合适的方案：

```
开始新 Agent 项目
    |
    v
是否需要极简依赖（嵌入式/无网络）？
    |                    |
   YES                  NO
    |                    |
    v                    v
从零实现             工具数量 > 5 个？
                        |              |
                       YES             NO
                        |              |
                        v              v
                   需要流式输出？    从零实现即可
                    |        |       （简单场景）
                   YES      NO
                    |        |
                    v        v
                LangChain   工具需要
                框架       复杂状态管理？
                            |        |
                           YES      NO
                            |        |
                            v        v
                        LangChain  从零实现
                        框架       （轻量场景）
```

**快速判断口诀：**
- 工具多（>5） + 状态复杂 + 需要生产特性 -> **LangChain**
- 工具少（<=5） + 逻辑简单 + 资源受限 -> **从零实现**
- 学习研究 -> **从零实现**
- 生产部署 -> **LangChain**

## 7. 高级: 自定义中间件 with_config()

LangChain 允许通过 `with_config()` 注入自定义行为：
- 执行前后的钩子函数
- 自定义重试策略
- 速率限制
- 日志和监控

### 7.1 监控中间件

In [ ]:
# ============================================================
# 7.1 监控中间件
# ============================================================

from langchain_core.runnables import RunnableLambda, RunnableConfig
from langchain_core.callbacks import BaseCallbackHandler


class AgentMonitorCallback(BaseCallbackHandler):
    """
    自定义回调处理器 —— 监控 Agent 执行过程。
    追踪 LLM 调用次数、工具调用记录、执行时间。
    """

    def __init__(self):
        super().__init__()
        self.tool_calls = []
        self.llm_calls = 0
        self.start_time = None

    def on_llm_start(self, serialized, prompts, **kwargs):
        self.llm_calls += 1
        if self.start_time is None:
            self.start_time = time.time()

    def on_tool_start(self, serialized, input_str, **kwargs):
        tool_name = serialized.get("name", "unknown")
        self.tool_calls.append({
            "tool": tool_name,
            "input": str(input_str)[:100],
            "timestamp": time.time(),
        })

    def get_report(self) -> str:
        elapsed = time.time() - self.start_time if self.start_time else 0
        return (
            f"Agent 执行报告:\n"
            f"  LLM 调用次数: {self.llm_calls}\n"
            f"  工具调用次数: {len(self.tool_calls)}\n"
            f"  总执行时间: {elapsed:.2f}s\n"
            f"  调用工具: {[tc['tool'] for tc in self.tool_calls]}"
        )


monitor = AgentMonitorCallback()

if HAS_LANGCHAIN:
    monitored_agent = agent.with_config({
        "callbacks": [monitor],
        "recursion_limit": 10,
        "tags": ["production", "monitored"],
    })
    print("[OK] 带监控的 Agent 配置完成")
else:
    print("[模拟] 带监控的 Agent 配置完成")
    print("使用 with_config() 可以注入:")
    print("  - callbacks: 自定义回调处理器")
    print("  - recursion_limit: 递归深度限制")
    print("  - tags: 执行标签(用于LangSmith追踪)")
    print("  - metadata: 自定义元数据")

### 7.2 重试中间件

通过 `RunnableLambda` 包装实现自动重试：当工具调用失败（可重试错误）时，自动重新执行。

In [ ]:
# ============================================================
# 7.2 重试中间件
# ============================================================

class RetryMiddleware:
    """
    重试中间件：包装 Runnable，自动对可重试错误进行重试。

    使用 with_retry() 方法包装任意 Runnable。
    支持指数退避和最大重试次数。
    """

    RETRYABLE_ERRORS = (
        "timeout", "connection", "rate limit", "temporary",
        "503", "502", "429", "too many requests",
    )

    def __init__(self, max_retries: int = 3, base_delay: float = 1.0):
        self.max_retries = max_retries
        self.base_delay = base_delay
        self.retry_count = 0

    def is_retryable(self, error_message: str) -> bool:
        """判断错误是否可重试"""
        msg_lower = error_message.lower()
        return any(indicator in msg_lower for indicator in self.RETRYABLE_ERRORS)

    def wrap_tool(self, tool_func: Callable) -> Callable:
        """
        包装工具函数，添加自动重试能力。

        返回包装后的函数，具有相同的签名和自动重试行为。
        """
        import functools

        @functools.wraps(tool_func)
        def wrapped(*args, **kwargs):
            last_error = None
            for attempt in range(self.max_retries + 1):
                try:
                    result = tool_func(*args, **kwargs)
                    # 检查结果中是否包含错误
                    if isinstance(result, str) and "错误" in result:
                        if self.is_retryable(result):
                            last_error = result
                            if attempt < self.max_retries:
                                delay = self.base_delay * (2 ** attempt)
                                time.sleep(delay)
                                self.retry_count += 1
                                continue
                    return result
                except Exception as e:
                    last_error = str(e)
                    if not self.is_retryable(str(e)):
                        raise
                    if attempt < self.max_retries:
                        delay = self.base_delay * (2 ** attempt)
                        time.sleep(delay)
                        self.retry_count += 1
            raise RuntimeError(f"重试耗尽 ({self.max_retries}次): {last_error}")

        return wrapped

    def get_stats(self) -> dict:
        return {"total_retries": self.retry_count, "max_retries": self.max_retries}


# 演示重试中间件
print("=" * 60)
print("重试中间件演示")
print("=" * 60)

# 模拟一个会间歇失败的工具
call_counter = {"count": 0}


def flaky_search_tool(query: str) -> str:
    """模拟一个不稳定的搜索工具：前两次超时，第三次成功"""
    call_counter["count"] += 1
    if call_counter["count"] <= 2:
        return f"错误: timeout - 搜索 '{query}' 超时"
    return f"搜索 '{query}' 成功: 找到 3 条结果"


# 创建重试中间件
retry_mw = RetryMiddleware(max_retries=2, base_delay=0.1)

# 包装工具
robust_search = retry_mw.wrap_tool(flaky_search_tool)

# 测试
result = robust_search("AI Agent")
print(f"\n最终结果: {result}")
print(f"重试统计: {retry_mw.get_stats()}")
print(f"\n说明: 工具前2次返回超时错误，中间件自动重试，第3次成功")

print("\n使用方式:")
print("  rmw = RetryMiddleware(max_retries=3, base_delay=1.0)")
print("  robust_tool = rmw.wrap_tool(original_tool)")
print("  # robust_tool 现在自动重试可恢复的错误")

## 8. 解析错误处理演示

当 Agent 输出格式不正确的工具调用时，需要优雅地处理。
以下实现一个多层回退的鲁棒解析器：JSON -> 正则 (括号) -> 正则 (方括号)。

In [ ]:
# ============================================================
# 鲁棒工具调用解析器（多层回退）
# ============================================================

class RobustToolCallParser:
    """
    鲁棒的工具调用解析器。

    三层回退策略:
    Layer 1: JSON 解析（最标准）
    Layer 2: 正则提取 tool_name(args) 模式
    Layer 3: 正则提取 tool_name[args] 模式
    全部失败: 返回详细错误信息 + 模糊匹配建议
    """

    def __init__(self, tools: list):
        self.tool_names = {t.name: t for t in tools}
        self._name_index = [t.name.lower() for t in tools]

    def parse(self, raw_text: str) -> tuple:
        """
        多层解析策略。
        返回 (tool_name, kwargs, success, error_message)
        """
        # Layer 1: JSON 解析
        try:
            data = json.loads(raw_text)
            if "tool" in data:
                return (data["tool"], data.get("args", {}), True, "")
        except json.JSONDecodeError:
            # 不是 JSON，继续尝试其他格式
            pass

        # Layer 2: tool_name(args) 模式
        match = re.search(r'(\w+)\s*\(\s*(.+?)\s*\)', raw_text)
        if match:
            tool_name = match.group(1)
            args_str = match.group(2)
            if tool_name not in self.tool_names:
                similar = self._find_similar(tool_name)
                if similar:
                    return (similar, {}, True, f"自动修正: '{tool_name}' -> '{similar}'")
                return (None, {}, False, f"工具 '{tool_name}' 不存在")
            kwargs = self._parse_kv_args(args_str)
            return (tool_name, kwargs, True, "")

        # Layer 3: tool_name[args] 模式
        match = re.search(r'(\w+)\s*\[\s*(.+?)\s*\]', raw_text)
        if match:
            tool_name = match.group(1)
            args_str = match.group(2)
            kwargs = self._parse_kv_args(args_str)
            return (tool_name, kwargs, True, "")

        return (None, {}, False, f"无法解析工具调用: {raw_text[:100]}")

    def _parse_kv_args(self, text: str) -> dict:
        """解析 key=value 参数对，支持引号和数字类型推断"""
        kwargs = {}
        for match in re.finditer(
            r'([a-zA-Z_][a-zA-Z0-9_]*)\s*=\s*'
            r'(?:"([^"]*)"|\'([^\']*)\'|([^,\)\]]+))',
            text
        ):
            key = match.group(1)
            value = match.group(2) or match.group(3) or match.group(4)
            value = value.strip()
            if value.isdigit():
                value = int(value)
            elif value.lower() in ('true', 'false'):
                value = value.lower() == 'true'
            kwargs[key] = value
        return kwargs

    def _find_similar(self, name: str) -> str:
        """查找最相似的工具名（子串匹配）"""
        name_l = name.lower()
        for registered in self.tool_names:
            if name_l in registered.lower() or registered.lower() in name_l:
                return registered
        return ""


# 测试解析器
parser = RobustToolCallParser(tools)

test_outputs = [
    'weather_tool(city="北京", units="celsius")',
    'calculator_tool(formula=2+2*3)',
    '{"tool": "search_tool", "args": {"query": "AI"}}',
    'weather_tool[city=上海]',
    'get_weather(location=北京)',      # 工具名不存在 -> 模糊匹配
    'calc(expression=2+2)',           # 别名 -> 模糊匹配
]

print("=" * 60)
print("鲁棒工具调用解析器测试")
print("=" * 60)

for raw in test_outputs:
    tool_name, kwargs, success, msg = parser.parse(raw)
    status = "OK" if success else "FAIL"
    print(f"\n[{status}] 输入: {raw[:60]}...")
    print(f"  -> 工具: {tool_name}, 参数: {kwargs}")
    if msg:
        print(f"  消息: {msg}")

## 9. 总结

### 本 Notebook 涵盖的内容

| 章节 | 内容 | 关键技能 |
|------|------|----------|
| 1 | 环境准备 | LangChain 依赖安装 |
| 2 | @tool 装饰器 | 创建 4+ 个 LangChain 风格工具 |
| 3 | create_react_agent | 声明式构建 ReAct Agent |
| 4 | AgentExecutor | 执行控制 + handle_parsing_errors 三种模式 |
| 5 | 研究型 Agent | 多工具综合应用完整示例 |
| 6 | 框架对比 + 决策树 | 从零 vs 框架：代码量/功能/场景 |
| 7 | 自定义中间件 | 监控回调 + 重试中间件 + with_config |
| 8 | 解析错误处理 | 三层回退鲁棒解析器 |

### 关键要点

- **学习时从零实现**：理解 Agent 的底层工作原理（参见 Phase 06-02）
- **生产时使用框架**：获得追踪、监控、错误恢复等基础设施
- **工具描述是关键**：好的 docstring 直接影响 Agent 的工具选择准确性
- **永远设置执行限制**：max_iterations 和 max_time 防止失控
- **解析错误必须处理**：使用多层回退策略提高鲁棒性
- **决策树**：工具多 + 状态复杂 + 生产需求 -> LangChain；工具少 + 简单 + 受限 -> 从零实现